# EXP-005: Driver Feature Engineering - Playground Series S6E5

**근거:** EXP-004에서 단일 모델이 모두 +0.0003씩 개선되며 다양성 ↓ → 블렌딩 LB 전이율 25%로 떨어짐. Driver 887 카디널리티는 명백한 미사용 자원. 새 시그널 도입으로 다양성 회복 + 단일 모델 개선 시도.

**FE 두 가지:**
1. `Driver_freq` — train 전체로 계산한 등장 횟수. 라벨 무관이라 누수 없음.
2. `Driver_te` — **fold 내 train만**으로 계산한 smoothed target encoding. 누수 방지가 핵심.
   - Smoothing: `(n * mean + prior_w * prior) / (n + prior_w)`, prior_w=10, prior=전체 평균
   - Test에는 5-fold TE의 평균 적용

**나머지 설정**: EXP-004와 동일 (lr=0.02, n_est 확장, GPU XGB/CAT, CPU LGB)

In [1]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

train = pd.read_csv('../data/train.csv')
test  = pd.read_csv('../data/test.csv')

TARGET = 'PitNextLap'
CAT_COLS = ['Driver', 'Compound', 'Race']
y = train[TARGET].astype(int).values
GLOBAL_MEAN = y.mean()
print(f'Train: {train.shape}, Test: {test.shape}, global pos rate: {GLOBAL_MEAN:.4f}')

Train: (439140, 16), Test: (188165, 15), global pos rate: 0.1990


## 1. Driver Frequency Encoding (누수 없음)

In [2]:
# 라벨 무관 통계라 train 전체로 계산 OK
freq_map = train['Driver'].value_counts().to_dict()
train['Driver_freq'] = train['Driver'].map(freq_map).astype(np.float32)
test['Driver_freq']  = test['Driver'].map(freq_map).fillna(0).astype(np.float32)

print(f'Driver freq stats — train: mean {train["Driver_freq"].mean():.1f}, max {train["Driver_freq"].max():.0f}')
print(f'                  test : mean {test["Driver_freq"].mean():.1f}, missing {test["Driver_freq"].isna().sum()}')

Driver freq stats — train: mean 1148.2, max 1682
                  test : mean 1147.3, missing 0


## 2. Driver Target Encoding (fold-wise, smoothed)

각 fold의 train만으로 driver별 평균 PitNextLap 계산. val/test에 매핑.
- val: 해당 fold의 train으로 fit한 인코딩
- test: 5-fold의 평균 (모든 fold가 동등 기여)

In [3]:
N_FOLDS, SEED = 5, 42
PRIOR_W = 10  # smoothing factor

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
splits = list(skf.split(train, y))

train['Driver_te'] = np.nan
test_te = np.zeros(len(test), dtype=np.float32)

for fold, (tr_idx, va_idx) in enumerate(splits):
    fold_tr = train.iloc[tr_idx]
    # smoothed mean per driver
    grp = fold_tr.groupby('Driver').agg(n=(TARGET, 'size'), mean=(TARGET, 'mean'))
    grp['te'] = (grp['n'] * grp['mean'] + PRIOR_W * GLOBAL_MEAN) / (grp['n'] + PRIOR_W)
    te_map = grp['te'].to_dict()
    train.iloc[va_idx, train.columns.get_loc('Driver_te')] = train.iloc[va_idx]['Driver'].map(te_map).fillna(GLOBAL_MEAN).values
    test_te += test['Driver'].map(te_map).fillna(GLOBAL_MEAN).values / N_FOLDS

train['Driver_te'] = train['Driver_te'].astype(np.float32)
test['Driver_te']  = test_te.astype(np.float32)

print('Driver_te stats:')
print(f'  train mean {train["Driver_te"].mean():.4f}, std {train["Driver_te"].std():.4f}, range [{train["Driver_te"].min():.4f}, {train["Driver_te"].max():.4f}]')
print(f'  test  mean {test["Driver_te"].mean():.4f}, std {test["Driver_te"].std():.4f}')

# 누수 점검: val의 te가 그 row의 raw target과 직접 같은지
sample = train.sample(5, random_state=0)
print()
print('Sanity sample (te 값이 GLOBAL_MEAN 0.199 근처에 분산돼야 정상):')
print(sample[['Driver', TARGET, 'Driver_te']].to_string(index=False))

Driver_te stats:
  train mean 0.2001, std 0.0508, range [0.0167, 0.5759]
  test  mean 0.2001, std 0.0503

Sanity sample (te 값이 GLOBAL_MEAN 0.199 근처에 분산돼야 정상):
Driver  PitNextLap  Driver_te
  D050         0.0   0.184153
   GRO         0.0   0.230956
  D131         1.0   0.182993
   PIA         1.0   0.298057
  D037         1.0   0.218371


## 3. 학습 데이터 준비

In [4]:
# LE 버전 (XGB / LGB)
train_le = train.copy(); test_le = test.copy()
for c in CAT_COLS:
    le = LabelEncoder()
    train_le[c] = le.fit_transform(train_le[c].astype(str))
    seen = set(le.classes_)
    test_le[c] = test_le[c].astype(str).map(lambda v: le.transform([v])[0] if v in seen else -1)

# CAT 버전 (string 그대로)
train_cb = train.copy(); test_cb = test.copy()
for c in CAT_COLS:
    train_cb[c] = train_cb[c].astype(str)
    test_cb[c]  = test_cb[c].astype(str)

drop_cols = ['id', TARGET]
feature_cols = [c for c in train.columns if c not in drop_cols]
X_le, X_test_le = train_le[feature_cols], test_le[feature_cols]
X_cb, X_test_cb = train_cb[feature_cols], test_cb[feature_cols]
print(f'Features ({len(feature_cols)}): {feature_cols}')

LR = 0.02

Features (16): ['Driver', 'Compound', 'Race', 'Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', 'Driver_freq', 'Driver_te']


## 4. XGB (GPU, lr=0.02)

In [5]:
from xgboost import XGBClassifier

xgb_params = dict(
    n_estimators=5000, max_depth=8, learning_rate=LR,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
    gamma=0.1, reg_alpha=0.1, reg_lambda=1.0,
    objective='binary:logistic', eval_metric='auc',
    device='cuda', tree_method='hist',
    random_state=SEED, verbosity=0,
    early_stopping_rounds=200,
)

oof_xgb = np.zeros(len(X_le))
test_xgb = np.zeros(len(X_test_le))
xgb_iters = []
t0 = time.time()
for fold, (tr_idx, va_idx) in enumerate(splits):
    m = XGBClassifier(**xgb_params)
    m.fit(X_le.iloc[tr_idx], y[tr_idx],
          eval_set=[(X_le.iloc[va_idx], y[va_idx])], verbose=0)
    oof_xgb[va_idx] = m.predict_proba(X_le.iloc[va_idx])[:, 1]
    test_xgb += m.predict_proba(X_test_le)[:, 1] / N_FOLDS
    xgb_iters.append(m.best_iteration)
    print(f'XGB fold {fold}: AUC={roc_auc_score(y[va_idx], oof_xgb[va_idx]):.5f}, best_iter={m.best_iteration}')

xgb_oof_auc = roc_auc_score(y, oof_xgb)
print(f'XGB OOF AUC: {xgb_oof_auc:.5f}  (best_iter mean {np.mean(xgb_iters):.0f}, elapsed {time.time()-t0:.1f}s)')

XGB fold 0: AUC=0.95060, best_iter=2661
XGB fold 1: AUC=0.94863, best_iter=2144
XGB fold 2: AUC=0.94932, best_iter=2155
XGB fold 3: AUC=0.94885, best_iter=2573
XGB fold 4: AUC=0.94965, best_iter=2101
XGB OOF AUC: 0.94941  (best_iter mean 2327, elapsed 77.5s)


## 5. LGB (CPU, lr=0.02)

In [6]:
from lightgbm import LGBMClassifier, early_stopping, log_evaluation

lgb_params = dict(
    n_estimators=5000, learning_rate=LR, num_leaves=64, max_depth=-1,
    min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0,
    objective='binary', metric='auc',
    random_state=SEED, n_jobs=-1, verbose=-1,
)

oof_lgb = np.zeros(len(X_le))
test_lgb = np.zeros(len(X_test_le))
lgb_iters = []
t0 = time.time()
for fold, (tr_idx, va_idx) in enumerate(splits):
    m = LGBMClassifier(**lgb_params)
    m.fit(X_le.iloc[tr_idx], y[tr_idx],
          eval_set=[(X_le.iloc[va_idx], y[va_idx])],
          callbacks=[early_stopping(200, verbose=False), log_evaluation(0)])
    oof_lgb[va_idx] = m.predict_proba(X_le.iloc[va_idx])[:, 1]
    test_lgb += m.predict_proba(X_test_le)[:, 1] / N_FOLDS
    lgb_iters.append(m.best_iteration_)
    print(f'LGB fold {fold}: AUC={roc_auc_score(y[va_idx], oof_lgb[va_idx]):.5f}, best_iter={m.best_iteration_}')

lgb_oof_auc = roc_auc_score(y, oof_lgb)
print(f'LGB OOF AUC: {lgb_oof_auc:.5f}  (best_iter mean {np.mean(lgb_iters):.0f}, elapsed {time.time()-t0:.1f}s)')

LGB fold 0: AUC=0.95024, best_iter=4805
LGB fold 1: AUC=0.94833, best_iter=3202
LGB fold 2: AUC=0.94904, best_iter=3155
LGB fold 3: AUC=0.94851, best_iter=3851
LGB fold 4: AUC=0.94959, best_iter=3850
LGB OOF AUC: 0.94914  (best_iter mean 3773, elapsed 126.1s)


## 6. CatBoost (GPU, lr=0.02)

In [7]:
from catboost import CatBoostClassifier

cat_idx = [feature_cols.index(c) for c in CAT_COLS]

cb_params = dict(
    iterations=20000, learning_rate=LR, depth=8, l2_leaf_reg=3.0,
    bagging_temperature=0.5, random_strength=1,
    eval_metric='AUC', loss_function='Logloss',
    cat_features=cat_idx, random_state=SEED,
    verbose=False, allow_writing_files=False,
    task_type='GPU', devices='0',
    early_stopping_rounds=200,
)

oof_cat = np.zeros(len(X_cb))
test_cat = np.zeros(len(X_test_cb))
cat_iters = []
t0 = time.time()
for fold, (tr_idx, va_idx) in enumerate(splits):
    m = CatBoostClassifier(**cb_params)
    m.fit(X_cb.iloc[tr_idx], y[tr_idx],
          eval_set=(X_cb.iloc[va_idx], y[va_idx]),
          use_best_model=True)
    oof_cat[va_idx] = m.predict_proba(X_cb.iloc[va_idx])[:, 1]
    test_cat += m.predict_proba(X_test_cb)[:, 1] / N_FOLDS
    cat_iters.append(m.get_best_iteration())
    print(f'CAT fold {fold}: AUC={roc_auc_score(y[va_idx], oof_cat[va_idx]):.5f}, best_iter={m.get_best_iteration()}')

cat_oof_auc = roc_auc_score(y, oof_cat)
print(f'CAT OOF AUC: {cat_oof_auc:.5f}  (best_iter mean {np.mean(cat_iters):.0f}, elapsed {time.time()-t0:.1f}s)')

Default metric period is 5 because AUC is/are not implemented for GPU


CAT fold 0: AUC=0.95057, best_iter=8831


Default metric period is 5 because AUC is/are not implemented for GPU


CAT fold 1: AUC=0.94862, best_iter=9149


Default metric period is 5 because AUC is/are not implemented for GPU


CAT fold 2: AUC=0.94951, best_iter=7208


Default metric period is 5 because AUC is/are not implemented for GPU


CAT fold 3: AUC=0.94893, best_iter=8166


Default metric period is 5 because AUC is/are not implemented for GPU


CAT fold 4: AUC=0.94991, best_iter=9939
CAT OOF AUC: 0.94950  (best_iter mean 8659, elapsed 2062.4s)


## 7. Blending

In [8]:
oof_blend_eq  = (oof_xgb + oof_lgb + oof_cat) / 3
test_blend_eq = (test_xgb + test_lgb + test_cat) / 3
auc_eq = roc_auc_score(y, oof_blend_eq)

best = (None, -1)
for w_xgb in np.arange(0, 1.001, 0.05):
    for w_lgb in np.arange(0, 1.001 - w_xgb, 0.05):
        w_cat = 1 - w_xgb - w_lgb
        if w_cat < 0: continue
        s = w_xgb * oof_xgb + w_lgb * oof_lgb + w_cat * oof_cat
        a = roc_auc_score(y, s)
        if a > best[1]:
            best = ((round(w_xgb, 2), round(w_lgb, 2), round(w_cat, 2)), a)

w_opt, auc_opt = best
oof_blend_opt  = w_opt[0] * oof_xgb + w_opt[1] * oof_lgb + w_opt[2] * oof_cat
test_blend_opt = w_opt[0] * test_xgb + w_opt[1] * test_lgb + w_opt[2] * test_cat

print('=== EXP-005 결과 (Driver FE 추가) ===')
print(f'XGB         OOF AUC: {xgb_oof_auc:.5f}')
print(f'LGB         OOF AUC: {lgb_oof_auc:.5f}')
print(f'CAT         OOF AUC: {cat_oof_auc:.5f}')
print(f'Blend equal       :  {auc_eq:.5f}')
print(f'Blend opt {w_opt}: {auc_opt:.5f}')
print()
print('=== vs EXP-004 (FE 없이) ===')
print(f'  XGB   0.94946 -> {xgb_oof_auc:.5f}  ({xgb_oof_auc-0.94946:+.5f})')
print(f'  LGB   0.94933 -> {lgb_oof_auc:.5f}  ({lgb_oof_auc-0.94933:+.5f})')
print(f'  CAT   0.94961 -> {cat_oof_auc:.5f}  ({cat_oof_auc-0.94961:+.5f})')
print(f'  Blend 0.95042 -> {auc_eq:.5f}      ({auc_eq-0.95042:+.5f})')
print(f'  Opt   0.95048 -> {auc_opt:.5f}     ({auc_opt-0.95048:+.5f})')

=== EXP-005 결과 (Driver FE 추가) ===
XGB         OOF AUC: 0.94941
LGB         OOF AUC: 0.94914
CAT         OOF AUC: 0.94950
Blend equal       :  0.95025
Blend opt (np.float64(0.3), np.float64(0.25), np.float64(0.45)): 0.95030

=== vs EXP-004 (FE 없이) ===
  XGB   0.94946 -> 0.94941  (-0.00005)
  LGB   0.94933 -> 0.94914  (-0.00019)
  CAT   0.94961 -> 0.94950  (-0.00011)
  Blend 0.95042 -> 0.95025      (-0.00017)
  Opt   0.95048 -> 0.95030     (-0.00018)


## 8. Submissions

In [9]:
for name, prob in [
    ('exp005_xgb',         test_xgb),
    ('exp005_lgb',         test_lgb),
    ('exp005_cat',         test_cat),
    ('exp005_blend_equal', test_blend_eq),
    ('exp005_blend_opt',   test_blend_opt),
]:
    sub = pd.DataFrame({'id': test['id'], 'PitNextLap': prob})
    path = f'../submissions/submission_{name}.csv'
    sub.to_csv(path, index=False)
    print(f'  saved {path}  mean={prob.mean():.4f}')

  saved ../submissions/submission_exp005_xgb.csv  mean=0.1971
  saved ../submissions/submission_exp005_lgb.csv  mean=0.1971
  saved ../submissions/submission_exp005_cat.csv  mean=0.1979
  saved ../submissions/submission_exp005_blend_equal.csv  mean=0.1974
  saved ../submissions/submission_exp005_blend_opt.csv  mean=0.1974
